# 03 — Pressure: Stay or Go?

PFF pass-rush micro-analysis. This notebook works directly off `pffScoutingData.csv` and is **independent of Sid's `route_features.parquet`** handoff — it never reads any route-feature parquet. Shared constants are imported from `src/config.py` by their actual names (config is Sid-owned and is not modified here).

In [ ]:
# --- imports & reliable project-relative paths ---
import sys
from pathlib import Path

# Locate project root robustly whether cwd is repo root or notebooks/
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / 'src' / 'config.py').exists():
        sys.path.insert(0, str(_p / 'src'))
        _ROOT = _p
        break
else:
    raise SystemExit('Could not locate src/config.py from cwd.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Shared constants imported by their actual names (Sid's config, unmodified).
from config import (
    DATA_DIR, ROOT, TE_RB_POSITIONS, RB_POSITIONS,
    END_EVENTS, NO_THROW_EVENTS, FPS,
)
print('project root:', ROOT)


## Input checks

In [ ]:
PFF_PATH = DATA_DIR / 'pffScoutingData.csv'
assert PFF_PATH.exists(), f'Missing required input: {PFF_PATH}'
print('PFF scouting data:', PFF_PATH)


## Load PFF scouting data

In [ ]:
pff = pd.read_csv(PFF_PATH, dtype={'gameId': 'Int64', 'playId': 'Int64', 'nflId': 'Int64'})
print('rows:', len(pff))
print('columns:', list(pff.columns))
pff['pff_role'].value_counts(dropna=False)


## Pass-rushers per play

Count distinct pass-rushers per `(gameId, playId)`. This is the denominator for any 'pressure vs. extra rusher' analysis. (Earlier inspection conflated the `Pass` role count — one QB per play — with the `Pass Rush` rows; they are not the same.)

In [ ]:
pr = pff[pff['pff_role'] == 'Pass Rush'].copy()
print('Pass Rush rows:', len(pr))
print('unique plays with pass rush:', pr.groupby(['gameId','playId']).ngroups)
print('duplicate (gameId,playId,nflId):', int(pr.duplicated(['gameId','playId','nflId']).sum()))
print('missing nflId:', int(pr['nflId'].isna().sum()))

rusher_counts = (pr.drop_duplicates(['gameId','playId','nflId'])
                   .groupby(['gameId','playId'])['nflId'].nunique())
dist = rusher_counts.value_counts().sort_index()
print('--- rusher-count distribution ---')
print(dist.to_string())
print('plays with >=5 rushers:', int((rusher_counts >= 5).sum()))


## Rusher-count distribution (figure)

In [ ]:
fig, ax = plt.subplots(figsize=(6,3.2))
dist.plot.bar(ax=ax, color='#444')
ax.set_xlabel('rushers per play')
ax.set_ylabel('plays')
ax.set_title('Pass-rushers per play (PFF)')
plt.tight_layout()
out = ROOT / 'figures' / 'rushers_per_play.png'
out.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out, dpi=120)
plt.show()
print('saved:', out)


## Pressure production by rusher count

Exploratory only — share of plays with any hit/hurry/sack, split by number of rushers. Not a final result.

In [ ]:
prx = pr.groupby(['gameId','playId']).agg(
    n_rushers=('nflId', 'nunique'),
    any_hit=('pff_hit', lambda s: int(s.fillna(0).gt(0).any())),
    any_hurry=('pff_hurry', lambda s: int(s.fillna(0).gt(0).any())),
    any_sack=('pff_sack', lambda s: int(s.fillna(0).gt(0).any())),
).reset_index()
prx['any_pressure'] = prx[['any_hit','any_hurry','any_sack']].max(axis=1)
tbl = prx.groupby('n_rushers')['any_pressure'].agg(['count','mean']).rename(
    columns={'count':'plays','mean':'pressure_rate'})
print(tbl.to_string())


## Next steps / open questions

- Snap and endpoint event ordering when multiple autoevents fire needs confirming with Sid (`SNAP_EVENTS` / `END_EVENTS` in config) before timing pressure windows.
- Does `NO_THROW_EVENTS` (qb_sack / qb_strip_sack / run) correctly mark no-throw plays for the stay-or-go split?
- This notebook is intentionally independent of `route_features.parquet`; pressure may later be joined to route outcomes once Sid's handoff lands.

> Event-handling questions go to Sid. Do not assume every autoevent is interchangeable without checking endpoint precedence.